# Inter Annotator Agreement (IAA) with gitma
This demo uses the demo CATMA project.
If you want to use it for your own annotations you first have to clone your CATMA project locally.
For further information about cloning your CATMA project see [this notebook](https://github.com/forTEXT/gitma/blob/main/demo/notebooks/load_project_from_gitlab.ipynb).

This package provides three methods to compute the agreement of two annotators.
All three methods compare annotation collections.
For that reason, it is recommended to use one annotation collection per annotator and document.
Additionally, it is recommended to name every annotation collection by a combination of the <span style="color:pink">document's title</span>, the <span style="color:red">annotation task</span> and the <span style="color:green">annotator</span>.

**Example:**  <span style="color:pink">robinson_crusoe</span>-<span style="color:red">narrative_space</span>-<span style="color:green">mareike</span>

## Table of contents
* [Dependencies](#1-bullet)
* [Load a CATMA project](#2-bullet)
* [`calculate_scotts_pi()` and `calculate_cohens_kappa()`](#3-bullet)
  * [Basic example](#3.1)
  * [Filter by tags](#3.2)
  * [Compare annotation properties](#3.3)
* [`calculate_krippendorffs_alpha()`](#4-bullet)
  * [Basic example](#4.1)
  * [Filter by tags](#4.2)
* [`gamma_agreement()`](#5-bullet)

## Dependencies <a class="anchor" id="1-bullet"></a>

### nltk

If you are only interested in IAA metrics such as *Scott's pi*, *Cohen's kappa* and *Krippendorff's alpha*
the [Natural Language Toolkit](https://www.nltk.org/) is sufficient (already installed).

### pygamma-agreement

The gamma agreement takes unitizing as part of annotation tasks into account
(see [Mathet et al. 2015](https://aclanthology.org/J15-3003.pdf)).
For many annotation projects done with CATMA, that might be crucial.
If you want to compute the gamma agreement using this package, the installation of [pygamma-agreement](https://github.com/bootphon/pygamma-agreement) is required
(already installed if you are using the GitMA Docker image):

    pip install pygamma-agreement==0.5.6

Please take note of the **further installation instructions** on the [pygamma-agreement GitHub page](https://github.com/bootphon/pygamma-agreement#installation) and the [*'how to cite'*](https://github.com/bootphon/pygamma-agreement#citing-pygamma)!

## Load a CATMA project <a class="anchor" id="2-bullet"></a>

In [ ]:
from gitma import CatmaProject

my_project = CatmaProject(
    projects_directory='../projects/',
    project_name='GitMA_Demo_Project',
)

*The above code block loads the included demo project. If you would like to try the examples below with one of your own projects that you previously loaded using the `load_project_from_gitlab` notebook,
uncomment the code in the following block (remove the leading hashes) and fill in your project name. Then execute that block instead of the one above.
Note, however, that many of the parameters in the examples below (such as tag, property and annotation collection names) are specific to the demo project and will likely also need to be modified.*

In [ ]:
#from gitma import CatmaProject

#my_project = CatmaProject(
#    projects_directory='../../user_projects/',
#    project_name='insert your project name here'
#)

## Calculate Scott's pi and Cohen's kappa for two annotators with `calculate_scotts_pi()` and `calculate_cohens_kappa()` <a class="anchor" id="3-bullet"></a>


The demo project contains three annotation collections.
In this demo we will compute the agreement of the collections 'ac_1' and 'ac_2'.

For every annotation in annotation collection 1 (`ac1_name_or_inst`) both methods search for the best matching annotation
in annotation collection 2 (`ac2_name_or_inst`) with respect to its annotation text span.
The following examples show how matching annotations in two annotation collections are identified:

<img src="img/best_match_example_iaa.png">

In contrast to the `gamma_agreement` method (see below), both `calculate_scotts_pi()` and `calculate_cohens_kappa()` only consider the best matching annotations
from both annotation collections when computing the IAA value.

### Basic example <a class="anchor" id="3.1"></a>

First, we will take look at both annotation collections by comparing the annotation spans.

In [ ]:
# compare the annotation collections by start point
my_project.compare_annotation_collections(
    annotation_collections=['ac_1', 'ac_2']
)

As the line plot shows, every annotation in annotation collection 'ac_1' has a matching annotation in annotation collection 'ac_2'.

Now, let's compute the Scott's pi IAA score for all matching annotations:

In [ ]:
pi, confusion_matrix = my_project.calculate_scotts_pi(
    ac1_name_or_inst='ac_1',
    ac2_name_or_inst='ac_2'
)

Next, let's compute the Cohens's Kappa IAA score for all matching annotations:

In [ ]:
kappa, confusion_matrix = my_project.calculate_cohens_kappa(
    ac1_name_or_inst='ac_1',
    ac2_name_or_inst='ac_2'
)

Both methods not only return agreement scores,
but also report the number of annotation pairs considered when computing the IAA score
and the average overlap of the annotation pairs.
Additionally, both methods return a confusion matrix to give an insight into the relation between the tags.
As you can see in the matrices, in 2 cases an annotation with the tag 'non_event' in annotation collection 1
has a best match in annotation collection 2 with the same tag.
Compare this with the line plot above.

### Filter by tags <a class="anchor" id="3.2"></a>

There may occur cases in which you don't want to include all annotations in the computing of
the IAA scores.
In those cases just use the `tag_filter` parameter, which expects a list of tag names.

In [ ]:
pi, confusion_matrix = my_project.calculate_scotts_pi(
    ac1_name_or_inst='ac_1',
    ac2_name_or_inst='ac_2',
    tag_filter=['process_event']
)

As the confusion matrix shows for the calculation of Scott's Pi, only the annotations with the tag 'process_event' have been taken into account.
If we would want to filter only the first collection and compare it with a second, unfiltered collection, we can use the parameter `filter_both_ac`:

In [ ]:
pi, confusion_matrix = my_project.calculate_scotts_pi(
    ac1_name_or_inst='ac_1',
    ac2_name_or_inst='ac_2',
    tag_filter=['process_event'],
    filter_both_ac=False
)

This can be helpful when exploring the matches and mismatches of different annotation collections. But be careful: since the calculation of IAA scores is always a based on a comparison of two or more annotation collections, filtering just one annotation collection and comparing it to a "full" annotation collection can influence the IAA score significantly.

### Compare annotation properties <a class="anchor" id="3.3"></a>

The tag is only one level of CATMA annotations.
If you want to compare annotations by their properties this is possible too.
In the demo project the annotations have the property 'representation_type' to evaluate if a speech
or mental event is referenced in the text:

In [ ]:
my_project.compare_annotation_collections(
    annotation_collections=['ac_1', 'ac_2'],
    color_col='prop:representation_type'
)

To compute the agreement of annotation properties you just need to use the `level` parameter:

In [ ]:
pi, confusion_matrix = my_project.calculate_scotts_pi(
    ac1_name_or_inst='ac_1',
    ac2_name_or_inst='ac_2',
    level='prop:representation_type'
)


## Calculate Krippendorff's alpha for more than two annotators with `calculate_krippendorffs_alpha()` <a class="anchor" id="4-bullet"></a>


To compute the Krippendorff's alpha agreement for two or more annotatation collections, you can use the `calculate_krippendorffs_alpha()` method.
Krippendorff's alpha can be calculated for two or more annotation collections. If you do not supply a list of annotation collections,
the method will use all annotation collections in the project.

The method returns the value for Krippendorff's alpha and a co-occurrence matrix of tag matches.

Analogue to the calculation of Scott's pi and Cohen's kappa, the best matching annotation spans are used for the calculation.

### Basic example <a class="anchor" id="4.1"></a>

First, we will take a look at all annotation collections by comparing the annotation spans.

In [ ]:
# compare the annotation collections by start point
my_project.compare_annotation_collections(
    annotation_collections=['ac_1', 'ac_2', 'gold_annotation']
)

Now, let's compute the Krippendorff's alpha score for all matching annotations:

In [ ]:
krippendorff, cooccurrence_matrix = my_project.calculate_krippendorffs_alpha(
    ac_names=['ac_1', 'ac_2', 'gold_annotation']
)

On a side note, you wouldn't usually compare the annotations of single annotators with the gold standard. This example here serves as a demonstration of the comparison of more than two annotation collections and therefore uses the `gold_annotation` collection.

### Filter by tags <a class="anchor" id="4.2"></a>

There may occur cases in which you don't want to include all annotations in the computing of
the IAA scores.
In those cases just use the `tag_filter` parameter, which expects a list of tag names.

In [ ]:
krippendorff, cooccurrence_matrix = my_project.calculate_krippendorffs_alpha(
    ac_names = ['ac_1', 'ac_2', 'gold_annotation'],
    tag_filter=['process_event']
)

## `gamma_agreement()` <a class="anchor" id="5-bullet"></a>

To compute the gamma agreement, in addition to the annotation collections, 5 further parameters
have to be defined.
The `alpha`, `beta` and `delta_empty` parameters are necessary to compute the
[`CombinedCategoricalDissimilarity`](https://github.com/bootphon/pygamma-agreement/blob/master/pygamma_agreement/dissimilarity.py#L467).
The `n_samples` and the `precision_level` values are used in the 
[`compute_gamma()` method](https://github.com/bootphon/pygamma-agreement/blob/master/pygamma_agreement/continuum.py#L805).
See the documentation for pygamma-agreement and
[Mathet et al. 2015](https://aclanthology.org/J15-3003.pdf)
for further information about these parameters.

In [ ]:
# gamma agreement with default settings
my_project.gamma_agreement(
    annotation_collections=['ac_1', 'ac_2'],
    alpha=3,
    beta=1,
    delta_empty=0.01,
    n_samples=30,
    precision_level=0.01
)

If you want to work with a different dissimillarity algorithm,
consider using pygamma-agreement directly.
For this purpose you can save all annotations in a project as a CSV file
in the format pygamma-agreement takes as input:

In [ ]:
pygamma_df = my_project.pygamma_table(
    annotation_collections=['ac_1', 'ac_2']
)

# save
pygamma_df.to_csv('../pygamma_table.csv', index=False, header=False)

# show example
pygamma_df.head(5)